# Multi-Robot Path Finding: Cooperative A* vs POPS

This notebook compares:
1. **Fixed-priority Cooperative A\*** — plans robots in the input order.
2. **POPS (Priority-Order Portfolio Search)** — evaluates all robot priority permutations and selects the best complete schedule by makespan, then total path length.

The experiment varies **robot count** and **obstacle probability**, using the same generated grid, starts, and goals for both algorithms in each trial.

## Important
- POPS searches all priority orderings, but it does **not** search all possible joint paths and therefore does not guarantee globally optimal MAPF makespan.
- The default sweep uses one instance per configuration. For stronger conclusions, increase `REPEATS` to 3–5 if time permits.
- At 7 robots, POPS runs the prioritized planner 5,040 times per instance. This can take time.
- Makespan and total path length are recorded only when every robot succeeds. Success rate is reported separately.


In [ ]:
import os
import time
import random
import itertools
import math
from collections import deque

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from cooperative_astar import prioritized_cooperative_astar

# ---------------- EXPERIMENT SETTINGS ----------------
GRID_ROWS = 10
GRID_COLS = 10
ROBOT_COUNTS = [2, 3, 4, 5, 6, 7]
OBSTACLE_PROBS = [0.10, 0.20, 0.30]
REPEATS = 1
BASE_SEED = 42

RESULTS_DIR = "results"
os.makedirs(RESULTS_DIR, exist_ok=True)
OUTPUT_CSV = os.path.join(RESULTS_DIR, "cooperative_vs_pops_results.csv")

print("Experiment configuration loaded.", flush=True)
print("Configurations:", len(ROBOT_COUNTS) * len(OBSTACLE_PROBS) * REPEATS)
print("At 7 robots, POPS evaluates all 7! = 5,040 priority orders per instance.", flush=True)


## 1. Generate reproducible test instances

Every algorithm is evaluated on the exact same instance for a given robot count, obstacle probability, and repeat. Each start and goal has an individual path through free cells; this does not guarantee that a collision-free multi-robot schedule exists.


In [ ]:
def individually_reachable(grid, start, goal):
    rows, cols = grid.shape
    queue = deque([start])
    visited = {start}
    moves = [(-1, 0), (1, 0), (0, -1), (0, 1)]
    while queue:
        r, c = queue.popleft()
        if (r, c) == goal:
            return True
        for dr, dc in moves:
            nr, nc = r + dr, c + dc
            nxt = (nr, nc)
            if (0 <= nr < rows and 0 <= nc < cols
                    and grid[nr, nc] == 0 and nxt not in visited):
                visited.add(nxt)
                queue.append(nxt)
    return False


def generate_instance(rows, cols, obstacle_prob, n_robots, seed, max_attempts=1000):
    rng = np.random.default_rng(seed)
    py_rng = random.Random(seed)
    total_cells = rows * cols
    for _ in range(max_attempts):
        grid = rng.choice([0, 1], size=(rows, cols),
                          p=[1 - obstacle_prob, obstacle_prob]).astype(int)
        chosen = py_rng.sample(range(total_cells), 2 * n_robots)
        starts = [(p // cols, p % cols) for p in chosen[:n_robots]]
        goals = [(p // cols, p % cols) for p in chosen[n_robots:]]
        for cell in starts + goals:
            grid[cell] = 0
        if all(individually_reachable(grid, starts[i], goals[i])
               for i in range(n_robots)):
            return grid, starts, goals
    raise RuntimeError(
        f"Unable to generate instance for p={obstacle_prob}; "
        "try reducing obstacle probability or increasing grid size."
    )


def plot_instance(grid, starts, goals, title="MAPF test instance"):
    fig, ax = plt.subplots(figsize=(6, 6))
    ax.imshow(grid, cmap="Greys", vmin=0, vmax=1)
    ax.set_xticks(range(grid.shape[1]))
    ax.set_yticks(range(grid.shape[0]))
    ax.grid(True, linewidth=0.4)
    for i, (s, g) in enumerate(zip(starts, goals), start=1):
        ax.scatter(s[1], s[0], marker="o", s=80, label=f"R{i} start")
        ax.scatter(g[1], g[0], marker="*", s=130, label=f"R{i} goal")
        ax.text(s[1], s[0], str(i), ha="center", va="center", fontsize=8)
    ax.set_title(title)
    ax.invert_yaxis()
    ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=7)
    plt.tight_layout()
    plt.show()


## 2. Metrics

- **Success rate:** percentage of robots that reached their goals.
- **Makespan:** latest arrival time, calculated only if all robots succeed.
- **Total path length:** sum of robot path lengths, calculated only if all robots succeed.
- **Runtime:** includes all work performed by the algorithm. POPS runtime includes every priority permutation.
- **Nodes expanded:** sum of the low-level node-expansion counts returned by the planner.


In [ ]:
def summarize(algorithm, results, n_robots, elapsed, instance_id, obstacle_prob, details=""):
    robot_results = [results.get(f"Robot {i+1}", {}) for i in range(n_robots)]
    solved = sum(bool(r.get("success")) for r in robot_results)
    complete = solved == n_robots
    arrival_times = [
        r.get("arrival_time") for r in robot_results
        if r.get("success") and r.get("arrival_time") is not None
    ]
    path_lengths = [
        r.get("path_length") for r in robot_results
        if r.get("success") and r.get("path_length") is not None
    ]
    return {
        "instance_id": instance_id,
        "robots": n_robots,
        "obstacle_probability": obstacle_prob,
        "algorithm": algorithm,
        "robots_solved": solved,
        "success_rate_percent": round(100 * solved / n_robots, 2),
        "all_succeeded": complete,
        "makespan": max(arrival_times) if complete and arrival_times else np.nan,
        "total_path_length": sum(path_lengths) if complete else np.nan,
        "nodes_expanded": sum(r.get("nodes_expanded", 0) for r in robot_results),
        "runtime_seconds": round(elapsed, 6),
        "details": details
    }


def save_results(rows):
    pd.DataFrame(rows).to_csv(OUTPUT_CSV, index=False)


## 3. Run Cooperative A* and exhaustive POPS

The notebook prints progress while evaluating POPS. It saves the CSV after each algorithm so that finished results are preserved if a later run is interrupted.


In [ ]:
rows = []
instance_count = len(ROBOT_COUNTS) * len(OBSTACLE_PROBS) * REPEATS
instance_index = 0
sweep_start = time.perf_counter()

for obstacle_prob in OBSTACLE_PROBS:
    for n_robots in ROBOT_COUNTS:
        for repeat in range(REPEATS):
            instance_index += 1
            seed = BASE_SEED + int(obstacle_prob * 1000) * 100 + n_robots * 10 + repeat
            instance_id = f"p{obstacle_prob:.2f}_r{n_robots}_rep{repeat+1}"

            print(f"\n[{instance_index}/{instance_count}] {instance_id} (seed={seed})", flush=True)
            grid, starts, goals = generate_instance(
                GRID_ROWS, GRID_COLS, obstacle_prob, n_robots, seed
            )
            print(f"Grid={GRID_ROWS}x{GRID_COLS}; obstacles={int(grid.sum())}; robots={n_robots}", flush=True)

            # Cooperative A*: one fixed input order
            print("  Running fixed-priority Cooperative A*...", flush=True)
            start_time = time.perf_counter()
            coop_results = prioritized_cooperative_astar(grid, starts, goals)
            coop_elapsed = time.perf_counter() - start_time
            rows.append(summarize(
                "Cooperative A*", coop_results, n_robots, coop_elapsed,
                instance_id, obstacle_prob, "Fixed input priority order"
            ))
            save_results(rows)
            print(f"  Cooperative A* completed in {coop_elapsed:.3f}s", flush=True)

            # POPS: all possible priority orders for this robot count
            total_orders = math.factorial(n_robots)
            print(f"  Running POPS: testing all {n_robots}! = {total_orders:,} orders", flush=True)
            start_time = time.perf_counter()
            best_results = None
            best_key = None
            best_order = None
            complete_orders = 0

            for order_number, order in enumerate(itertools.permutations(range(n_robots)), start=1):
                trial = prioritized_cooperative_astar(
                    grid, starts, goals, priority_order=list(order)
                )
                robot_results = [trial.get(f"Robot {i+1}", {}) for i in range(n_robots)]
                solved = sum(bool(r.get("success")) for r in robot_results)
                complete = solved == n_robots
                arrivals = [
                    r.get("arrival_time") for r in robot_results
                    if r.get("success") and r.get("arrival_time") is not None
                ]
                lengths = [
                    r.get("path_length") for r in robot_results
                    if r.get("success") and r.get("path_length") is not None
                ]
                if complete:
                    complete_orders += 1
                makespan = max(arrivals) if complete and arrivals else float("inf")
                total_length = sum(lengths) if complete else float("inf")
                key = (
                    0 if complete else 1,
                    0 if complete else -solved,
                    makespan,
                    total_length
                )
                if best_key is None or key < best_key:
                    best_key, best_results, best_order = key, trial, order

                if order_number % 100 == 0 or order_number == total_orders:
                    print(f"    POPS progress: {order_number}/{total_orders}", flush=True)

            pops_elapsed = time.perf_counter() - start_time
            rows.append(summarize(
                "POPS", best_results, n_robots, pops_elapsed,
                instance_id, obstacle_prob,
                f"All {total_orders} orders tested; complete orders={complete_orders}; "
                f"best order={[i+1 for i in best_order] if best_order is not None else None}"
            ))
            save_results(rows)
            print(f"  POPS completed in {pops_elapsed:.3f}s; complete orders={complete_orders}", flush=True)

print(f"\nExperiment finished in {(time.perf_counter()-sweep_start)/60:.2f} minutes.")
results_df = pd.DataFrame(rows)
display(results_df)
print(f"Results saved to: {OUTPUT_CSV}")


## 4. Aggregate results and plot comparisons

Each plotted value is based on the configured number of repeats. With one repeat, use the plots as preliminary observations rather than general conclusions.


In [ ]:
results_df = pd.DataFrame(rows)
summary = (results_df.groupby(
    ["robots", "obstacle_probability", "algorithm"], as_index=False
).agg(
    success_rate_percent=("success_rate_percent", "mean"),
    makespan=("makespan", "mean"),
    total_path_length=("total_path_length", "mean"),
    runtime_seconds=("runtime_seconds", "mean")
))
display(summary)

for metric, ylabel in [
    ("makespan", "Makespan (timesteps; complete schedules only)"),
    ("runtime_seconds", "Runtime (seconds)"),
    ("success_rate_percent", "Robots solved (%)")
]:
    fig, ax = plt.subplots(figsize=(9, 5))
    for prob in OBSTACLE_PROBS:
        for algorithm in ["Cooperative A*", "POPS"]:
            part = summary[
                (summary["obstacle_probability"] == prob) &
                (summary["algorithm"] == algorithm)
            ].sort_values("robots")
            if not part.empty:
                ax.plot(
                    part["robots"], part[metric], marker="o",
                    label=f"{algorithm}, p={prob:.2f}"
                )
    ax.set_xlabel("Number of robots")
    ax.set_ylabel(ylabel)
    ax.set_title(f"{ylabel} vs robot count and obstacle probability")
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=8, ncol=2)
    plt.tight_layout()
    plt.show()

summary.to_csv(os.path.join(RESULTS_DIR, "cooperative_vs_pops_summary.csv"), index=False)
print("Summary saved to results/cooperative_vs_pops_summary.csv")


## 5. Interpretation notes

- POPS can match or outperform fixed-priority Cooperative A* because it tests alternative priority orders and chooses the best result among them.
- POPS is not guaranteed to find the globally optimal MAPF schedule: it exhaustively searches priority permutations, not all joint trajectories.
- Runtime is expected to increase with robot count because the number of permutations grows factorially.
- Use multiple random seeds per configuration for stronger conclusions. Keep the generated instance identical between algorithms for each comparison.
